# nb_fw_deploy_from_github

Publica os itens da pasta `/fabric` do repositório GitHub neste workspace pela API REST do Fabric (alternativa à Git integration).

1. `stage = "base"`: lakehouses com schemas, notebooks e pipeline.
2. Subir os CSVs em `lh_bronze/Files/landing/olist` e rodar o `pl_daily_load_olist`.
3. `stage = "model"`: modelo semântico Direct Lake e relatório.

In [ ]:
stage = "base"   # "base" ou "model"

In [ ]:
# Deploy the /fabric items of this repository into the current Fabric workspace, through the Fabric REST API.
#
# Alternative to Fabric Git integration, for tenants where GitHub sync is not enabled.
# Idempotent: creates missing items and updates the definition of existing ones.
#   stage = "base"  -> lakehouses (schemas enabled), notebooks, data pipeline
#   stage = "model" -> semantic model (Direct Lake bound to this workspace's lh_gold) and report
# Run "base", load the CSVs and run the pipeline (Gold tables must exist), then run "model".

import base64
import json
import time
from urllib.parse import quote

import requests
import sempy.fabric as fabric

REPOSITORY = "DanielResende23/fabric-lakehouse-medallion"
BRANCH = "main"
MODEL_NAME = "Olist Vendas e Perdas"
ZERO_GUID = "00000000-0000-0000-0000-000000000000"

client = fabric.FabricRestClient()
workspace_id = fabric.get_workspace_id()


def call(method: str, path: str, body: dict = None):
    response = client.request(method, path, json=body)
    if response.status_code == 202 and "x-ms-operation-id" in response.headers:
        operation_id = response.headers["x-ms-operation-id"]
        while True:
            time.sleep(3)
            operation = client.get(f"v1/operations/{operation_id}").json()
            if operation["status"] not in ("NotStarted", "Running"):
                break
        if operation["status"] != "Succeeded":
            raise RuntimeError(f"Operation {operation_id} failed: {operation}")
        result = client.get(f"v1/operations/{operation_id}/result")
        return result.json() if result.status_code == 200 and result.content else None
    if response.status_code >= 400:
        raise RuntimeError(f"{method} {path} -> {response.status_code}: {response.text}")
    return response.json() if response.content else None


def items() -> list:
    return call("GET", f"v1/workspaces/{workspace_id}/items")["value"]


def find(name: str, item_type: str):
    return next((i for i in items() if i["displayName"] == name and i["type"] == item_type), None)


tree = requests.get(f"https://api.github.com/repos/{REPOSITORY}/git/trees/{BRANCH}?recursive=1", timeout=60).json()["tree"]
repo_files = [t["path"] for t in tree if t["type"] == "blob" and t["path"].startswith("fabric/")]


def download(path: str) -> bytes:
    response = requests.get(f"https://raw.githubusercontent.com/{REPOSITORY}/{BRANCH}/{quote(path)}", timeout=60)
    response.raise_for_status()
    return response.content


def part(path: str, content: bytes) -> dict:
    return {"path": path, "payload": base64.b64encode(content).decode(), "payloadType": "InlineBase64"}


def folder_parts(folder: str, replace: dict = None) -> list:
    replace = replace or {}
    parts = []
    for path in repo_files:
        if not path.startswith(f"fabric/{folder}/") or path.endswith("/.platform") or "/.pbi/" in path:
            continue
        relative = path[len(f"fabric/{folder}/"):]
        parts.append(part(relative, replace[relative].encode() if relative in replace else download(path)))
    return parts


def platform(folder: str) -> dict:
    return json.loads(download(f"fabric/{folder}/.platform"))


def publish(name: str, item_type: str, parts: list, definition_format: str = None, description: str = "") -> str:
    definition = {"parts": parts}
    if definition_format:
        definition["format"] = definition_format
    existing = find(name, item_type)
    if existing:
        call("POST", f"v1/workspaces/{workspace_id}/items/{existing['id']}/updateDefinition", {"definition": definition})
        print(f"updated  {item_type:<14} {name}")
        return existing["id"]
    call("POST", f"v1/workspaces/{workspace_id}/items",
         {"displayName": name, "type": item_type, "description": description, "definition": definition})
    print(f"created  {item_type:<14} {name}")
    return find(name, item_type)["id"]


def deploy_base() -> None:
    for lakehouse in ("lh_bronze", "lh_silver", "lh_gold"):
        if find(lakehouse, "Lakehouse"):
            print(f"exists   {'Lakehouse':<14} {lakehouse}")
            continue
        call("POST", f"v1/workspaces/{workspace_id}/lakehouses", {
            "displayName": lakehouse,
            "description": platform(f"{lakehouse}.Lakehouse")["metadata"]["description"],
            "creationPayload": {"enableSchemas": True},
        })
        print(f"created  {'Lakehouse':<14} {lakehouse}")

    # logicalId (.platform) -> real item id, to rewrite the notebook references of the pipeline
    notebook_ids = {}
    notebook_folders = sorted({p.split("/")[1] for p in repo_files if p.split("/")[1].endswith(".Notebook")})
    for folder in notebook_folders:
        meta = platform(folder)
        content = download(f"fabric/{folder}/notebook-content.py")
        notebook_ids[meta["config"]["logicalId"]] = publish(
            folder[: -len(".Notebook")], "Notebook", [part("notebook-content.py", content)],
            "fabricGitSource", meta["metadata"]["description"])

    folder = "pl_daily_load_olist.DataPipeline"
    pipeline = download(f"fabric/{folder}/pipeline-content.json").decode()
    for logical_id, item_id in notebook_ids.items():
        pipeline = pipeline.replace(logical_id, item_id)
    pipeline = pipeline.replace(ZERO_GUID, workspace_id)
    publish("pl_daily_load_olist", "DataPipeline", [part("pipeline-content.json", pipeline.encode())],
            description=platform(folder)["metadata"]["description"])


def deploy_model() -> None:
    gold = find("lh_gold", "Lakehouse")
    if not gold:
        raise RuntimeError("lh_gold not found: run stage 'base' first")

    folder = f"{MODEL_NAME}.SemanticModel"
    expressions = download(f"fabric/{folder}/definition/expressions.tmdl").decode()
    expressions = expressions.replace(f"onelake.dfs.fabric.microsoft.com/{ZERO_GUID}/{ZERO_GUID}",
                                      f"onelake.dfs.fabric.microsoft.com/{workspace_id}/{gold['id']}")
    model_id = publish(MODEL_NAME, "SemanticModel",
                       folder_parts(folder, {"definition/expressions.tmdl": expressions}),
                       "TMDL", platform(folder)["metadata"]["description"])

    # Through the API a report references its model by id (byConnection), not by path
    folder = f"{MODEL_NAME}.Report"
    pbir = json.dumps({
        "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/report/definitionProperties/2.0.0/schema.json",
        "version": "4.0",
        "datasetReference": {"byConnection": {"connectionString": f"semanticmodelid={model_id}"}},
    }, indent=2)
    publish(MODEL_NAME, "Report", folder_parts(folder, {"definition.pbir": pbir}),
            "PBIR", platform(folder)["metadata"]["description"])

In [ ]:
deploy_base() if stage == "base" else deploy_model()